# BigGAN Multiple Classes: FFT Mean + XGBoost / MLP

Notebook ini disiapkan untuk eksperimen **multiple classes, single generator** pada subset `BigGAN/train`. Task tetap biner: `AI vs nature`.

Catatan penting: notebook ini **mengawali dengan audit subset**. Saat ini, eksperimen multiple classes yang sah mengharuskan `ai` dan `nature` sama-sama memuat beberapa kelas. Jika subset yang tersedia tidak memenuhi itu, notebook akan tetap membantu menunjukkan masalahnya sebelum training dilakukan.

In [ ]:
from pathlib import Path
import hashlib
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..")
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "genimage" / "BigGAN" / "train"
IMAGENET_MAP = PROJECT_ROOT / "data" / "imagenet_class_index.json"
MANIFEST_OUT = PROJECT_ROOT / "data" / "genimage_manifest_biggan_random5class_train.csv"
FFT_OUT = PROJECT_ROOT / "artifacts" / "features_fft_mean_biggan_random5class_train.csv"
METRICS_OUT = PROJECT_ROOT / "artifacts" / "results_classification_biggan_random5class_fft_only.csv"
PRED_OUT = PROJECT_ROOT / "artifacts" / "predictions_classification_biggan_random5class_fft_only.csv"

print("Data root:", DATA_ROOT.resolve())
print("ImageNet map:", IMAGENET_MAP.resolve())

## 1. Audit struktur subset

In [ ]:
ai_dir = DATA_ROOT / "ai"
nature_dir = DATA_ROOT / "nature"
assert ai_dir.exists(), f"Folder tidak ditemukan: {ai_dir}"
assert nature_dir.exists(), f"Folder tidak ditemukan: {nature_dir}"

ai_files = sorted([p for p in ai_dir.iterdir() if p.is_file()])
nature_files = sorted([p for p in nature_dir.iterdir() if p.is_file()])

print("AI files:", len(ai_files))
print("Nature files:", len(nature_files))

In [ ]:
mapping = json.loads(IMAGENET_MAP.read_text())
idx_to_info = {int(k): tuple(v) for k, v in mapping.items()}
wnid_to_label = {v[0]: v[1] for v in mapping.values()}

ai_counts = {}
for p in ai_files:
    m = re.match(r"(\d{3})_biggan_", p.name)
    if m:
        key = int(m.group(1))
        ai_counts[key] = ai_counts.get(key, 0) + 1

nature_counts = {}
for p in nature_files:
    m = re.match(r"(n\d{8})_", p.name)
    if m:
        key = m.group(1)
        nature_counts[key] = nature_counts.get(key, 0) + 1

ai_summary = pd.DataFrame([
    {"class_idx": idx, "wnid": idx_to_info[idx][0], "label": idx_to_info[idx][1], "count": count}
    for idx, count in sorted(ai_counts.items())
])

nature_summary = pd.DataFrame([
    {"wnid": wnid, "label": wnid_to_label.get(wnid, "UNKNOWN"), "count": count}
    for wnid, count in sorted(nature_counts.items())
])

print("AI class summary:")
display(ai_summary)
print("Nature class summary:")
display(nature_summary)

print("Unique AI classes:", len(ai_summary))
print("Unique nature classes:", len(nature_summary))

## 2. Interpretasi audit
Jika `nature` ternyata hanya memuat satu kelas, maka subset saat ini **belum cocok** untuk eksperimen multiple classes yang benar. Namun notebook tetap bisa dipakai untuk membangun manifest, memvisualisasikan data, dan menjalankan baseline sementara pada data yang tersedia.

In [ ]:
if len(ai_summary) < 2 or len(nature_summary) < 2:
    print("[WARN] Subset belum ideal untuk eksperimen multiple classes yang seimbang.")
    print("[WARN] Saat ini AI classes:", len(ai_summary), "| nature classes:", len(nature_summary))

## 3. Bangun manifest dari subset yang tersedia

In [ ]:
rows = []
for p in ai_files:
    image_id = "img_" + hashlib.sha1(str(p).encode("utf-8")).hexdigest()[:16]
    rows.append({
        "image_id": image_id,
        "path": str(p.resolve()),
        "relative_path": str(p.relative_to(DATA_ROOT)),
        "generator": "BigGAN",
        "subset_name": "random5class_train",
        "split": "train",
        "class_name": "ai",
        "content_id": p.name.split("_")[0],
        "is_real": 0,
        "y_ai": 1,
    })

for p in nature_files:
    m = re.match(r"(n\d{8})_", p.name)
    wnid = m.group(1) if m else "unknown"
    image_id = "img_" + hashlib.sha1(str(p).encode("utf-8")).hexdigest()[:16]
    rows.append({
        "image_id": image_id,
        "path": str(p.resolve()),
        "relative_path": str(p.relative_to(DATA_ROOT)),
        "generator": "BigGAN",
        "subset_name": "random5class_train",
        "split": "train",
        "class_name": "nature",
        "content_id": wnid,
        "is_real": 1,
        "y_ai": 0,
    })

manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFEST_OUT, index=False)
print("Saved manifest:", MANIFEST_OUT.resolve())
print("Manifest shape:", manifest.shape)
display(manifest.head())

## 4. Visualisasi sample image

In [ ]:
def show_grid(paths, title, ncols=4, figsize=(14, 7)):
    n = len(paths)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, path in zip(axes, paths):
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        ax.set_title(Path(path).name, fontsize=8)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_grid(manifest[manifest["y_ai"] == 1]["path"].head(8).tolist(), "AI samples")
show_grid(manifest[manifest["y_ai"] == 0]["path"].head(8).tolist(), "Nature samples")

## 5. Hitung fitur FFT mean

In [ ]:
def load_gray(path):
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32)

def extract_fft_mean(gray):
    f = np.fft.fft2(gray)
    mag = np.abs(f)
    phase = np.angle(f)
    return {
        "fft_mag_mean": float(np.mean(mag)),
        "fft_phase_mean": float(np.mean(phase)),
        "fft_phase_cos_mean": float(np.mean(np.cos(phase))),
        "fft_phase_sin_mean": float(np.mean(np.sin(phase))),
    }

rows = []
for row in manifest.itertuples(index=False):
    gray = load_gray(row.path)
    feats = extract_fft_mean(gray)
    rows.append({"image_id": row.image_id, **feats})

fft_df = pd.DataFrame(rows)
fft_df.to_csv(FFT_OUT, index=False)
print("Saved FFT features:", FFT_OUT.resolve())
print("FFT shape:", fft_df.shape)
display(fft_df.head())

## 6. Gabungkan manifest + fitur FFT

In [ ]:
data = manifest.merge(fft_df, on="image_id", how="inner")
feature_cols = ["fft_mag_mean", "fft_phase_mean", "fft_phase_cos_mean", "fft_phase_sin_mean"]
print(data.shape)
display(data.head())
display(data.groupby("y_ai")[feature_cols].agg(["mean", "std"]))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for ax, col in zip(axes, feature_cols):
    for label, color in [(1, "tab:red"), (0, "tab:blue")]:
        ax.hist(data[data["y_ai"] == label][col], bins=20, alpha=0.6, label=f"y_ai={label}", color=color)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Split 80:20 dan train baseline
Karena ini masih eksperimen awal pada subset train, evaluasi dilakukan dengan holdout 80:20.

In [ ]:
X = data[feature_cols].to_numpy(dtype=np.float32)
y = data["y_ai"].to_numpy(dtype=np.int64)

X_train, X_eval, y_train, y_eval, train_idx, eval_idx = train_test_split(
    X, y, data.index.to_numpy(), test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Eval:", X_eval.shape, y_eval.shape)

In [ ]:
models = {
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
    ),
    "MLP": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", max_iter=500, random_state=42)),
    ]),
}

results = []
pred_store = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    score_ai = model.predict_proba(X_eval)[:, 1]
    pred_ai = (score_ai >= 0.5).astype(np.int64)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_eval, pred_ai),
        "f1": f1_score(y_eval, pred_ai),
        "auroc": roc_auc_score(y_eval, score_ai),
    })
    pred_store[name] = {"score_ai": score_ai, "pred_ai": pred_ai}

results_df = pd.DataFrame(results).sort_values(by=["auroc", "f1", "accuracy"], ascending=False).reset_index(drop=True)
display(results_df)

## 8. Confusion matrix model terbaik

In [ ]:
best_model = results_df.iloc[0]["model"]
print("Best model:", best_model)
print()
print(classification_report(y_eval, pred_store[best_model]["pred_ai"], target_names=["nature", "ai"]))
cm = confusion_matrix(y_eval, pred_store[best_model]["pred_ai"])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["nature", "ai"]).plot(cmap="Blues")
plt.show()

## 9. Simpan hasil

In [ ]:
metrics_df = results_df.copy()
metrics_df["feature_set"] = "fft_mean_only"
metrics_df["split_protocol"] = "train_holdout_80_20"
metrics_df["subset_name"] = "biggan_random5class_train"
metrics_df["n_total"] = len(data)
metrics_df["n_train"] = len(X_train)
metrics_df["n_eval"] = len(X_eval)
metrics_df.to_csv(METRICS_OUT, index=False)

base_eval = data.iloc[eval_idx][["image_id", "path", "class_name", "content_id", "y_ai"]].copy()
pred_parts = []
for name, pred in pred_store.items():
    part = base_eval.copy()
    part["model"] = name
    part["score_ai"] = pred["score_ai"]
    part["pred_ai"] = pred["pred_ai"]
    pred_parts.append(part)

pred_df = pd.concat(pred_parts, ignore_index=True)
pred_df.to_csv(PRED_OUT, index=False)

print("Saved metrics:", METRICS_OUT.resolve())
print("Saved predictions:", PRED_OUT.resolve())
display(metrics_df)
display(pred_df.head())

## 10. Catatan interpretasi
Jika audit awal menunjukkan `nature` tidak benar-benar multi-class, maka hasil training di notebook ini harus diperlakukan sebagai **eksperimen sementara**, bukan eksperimen multiple classes yang final. Notebook ini tetap berguna untuk mengaudit subset, membangun manifest, dan mengukur baseline FFT pada data yang tersedia.